# Analysis of Results

NOT TO BE EDITED BY AGENTS

In [1]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go

## Visualize raw data

In [2]:
# Covid 19 US-NY cases & deaths
data = pd.read_csv("data/training/covid19_us/cases_deaths_US_New_York.csv")

In [3]:
data.date = pd.to_datetime(data.date)

In [4]:
data.head()

,country,province,date,case_cnt,death_cnt
0,US,New York,2020-03-08,105,0
1,US,New York,2020-03-09,138,0
2,US,New York,2020-03-10,152,0
3,US,New York,2020-03-11,216,1
4,US,New York,2020-03-12,296,1


In [5]:
data['active_cases'] = data['case_cnt'] - data['case_cnt'].shift(15)
data.fillna({'active_cases': 0}, inplace=True)

,country,province,date,case_cnt,death_cnt,active_cases
0,US,New York,2020-03-08,105,0,0.0
1,US,New York,2020-03-09,138,0,0.0
2,US,New York,2020-03-10,152,0,0.0
3,US,New York,2020-03-11,216,1,0.0
4,US,New York,2020-03-12,296,1,0.0
...,...,...,...,...,...,...
86,US,New York,2020-06-02,375887,30910,21387.0
87,US,New York,2020-06-03,376676,30964,20790.0
88,US,New York,2020-06-04,377888,31093,20836.0
89,US,New York,2020-06-05,379011,31120,20093.0


In [33]:
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=data["date"],
    y=data["case_cnt"],
    mode="lines",
    line=dict(width=2, color="#2a78d6"),
    hovertemplate="%{x|%b %d, %Y}<br>Cases: %{y:,}<extra></extra>",
))

fig.update_layout(
    title="COVID-19 Cases Over Time — US-NY",
    xaxis_title="Date",
    yaxis_title="Number of Cases",
    hovermode="x unified",
    template="plotly_white",
    plot_bgcolor="#fcfcfb",
    paper_bgcolor="#fcfcfb",
    xaxis=dict(showgrid=False, linecolor="#c3c2b7"),
    yaxis=dict(gridcolor="#e1e0d9", zeroline=False),
)

fig.show()

In [34]:
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=data["date"],
    y=data["active_cases"],
    mode="lines",
    line=dict(width=2, color="#2a78d6"),
    hovertemplate="%{x|%b %d, %Y}<br>Cases: %{y:,}<extra></extra>",
))

fig.update_layout(
    title="COVID-19 Active Cases — US-NY",
    xaxis_title="Date",
    yaxis_title="Number of Cases",
    hovermode="x unified",
    template="plotly_white",
    plot_bgcolor="#fcfcfb",
    paper_bgcolor="#fcfcfb",
    xaxis=dict(showgrid=False, linecolor="#c3c2b7"),
    yaxis=dict(gridcolor="#e1e0d9", zeroline=False),
)

fig.show()

In [35]:
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=data["date"],
    y=data["death_cnt"],
    mode="lines",
    line=dict(width=2, color="#e3a445"),
    hovertemplate="%{x|%b %d, %Y}<br>Deaths: %{y:,}<extra></extra>",
))

fig.update_layout(
    title="COVID-19 Deaths Over Time — US-NY",
    xaxis_title="Date",
    yaxis_title="Number of Deaths",
    hovermode="x unified",
    template="plotly_white",
    plot_bgcolor="#fcfcfb",
    paper_bgcolor="#fcfcfb",
    xaxis=dict(showgrid=False, linecolor="#c3c2b7"),
    yaxis=dict(gridcolor="#e1e0d9", zeroline=False),
)

fig.show()

## Set Training/Testing Data and Constants

We'll only consider the first wave of the pandemic for this experiment.

Heuristic: Select start date (first date at which cases > 100) and training cut-off (start date + 60 days)

In [6]:
from datetime import timedelta

In [7]:
start_date = data[data.case_cnt > 100].iloc[0]['date']
start_date

Timestamp('2020-03-08 00:00:00')

In [8]:
train_test_split = start_date + timedelta(days=60)
train_test_split

Timestamp('2020-05-07 00:00:00')

In [9]:
end_date = start_date + timedelta(days=90)
end_date

Timestamp('2020-06-06 00:00:00')

In [10]:
data = data[(data.date >= start_date) & (data.date <= end_date)].copy()

In [40]:
from plotly.subplots import make_subplots

fig = make_subplots(specs=[[{"secondary_y": True}]])
fig.add_trace(go.Scatter(
    x=data["date"],
    y=data["active_cases"],
    name="Active cases",
    mode="lines",
    line=dict(width=2, color="#2a78d6"),
    hovertemplate="%{x|%b %d, %Y}<br>Active cases: %{y:,}<extra></extra>",
), secondary_y=False)
fig.add_trace(go.Scatter(
    x=data["date"],
    y=data["death_cnt"],
    name="Deaths",
    mode="lines",
    line=dict(width=2, color="#e3a445"),
    hovertemplate="%{x|%b %d, %Y}<br>Deaths: %{y:,}<extra></extra>",
), secondary_y=True)

fig.add_vline(
    x=train_test_split,
    line_width=2,
    line_dash="dot",
    line_color="#d62728",
    annotation_text="Train/Test cutoff",
    annotation_position="top left",
)

fig.update_layout(
    title="COVID-19 Active Cases & Deaths — US-NY",
    xaxis_title="Date",
    hovermode="x unified",
    template="plotly_white",
    plot_bgcolor="#fcfcfb",
    paper_bgcolor="#fcfcfb",
    xaxis=dict(showgrid=False, linecolor="#c3c2b7"),
    legend=dict(orientation="h", y=1.08, x=0),
)
fig.update_yaxes(title_text="Active Cases", title_font_color="#2a78d6",
                 gridcolor="#e1e0d9", zeroline=False, secondary_y=False)
fig.update_yaxes(title_text="Deaths", title_font_color="#e3a445",
                 showgrid=False, zeroline=False, secondary_y=True)

fig.show()

## Plot Results

In [12]:
import json
import os
from train import forecast

Preparing the input data

In [ ]:
# selected region
region = 'New York'
best_params_filename = 'optimal_params_20261119_200602.json'

# split training data
data['time_index'] = (data["date"] - start_date).dt.days.astype(int)
T = np.array(data.time_index.values)
training_data = data[data.date <= train_test_split.to_pydatetime()].copy()

# load population for the region
population_us_states = pd.read_csv('data/training/Population_US_States.csv')
population = population_us_states[population_us_states.Province == region].iloc[0]['pop2016']

# load final model optimal parameters
all_optimal_params = json.load(open(f'data/model_params/{best_params_filename}', 'r'))
optimal_params = list(all_optimal_params[region].values())

Run forecast

In [32]:
cases_forecast, deaths_forecast = forecast(training_data, optimal_params, T, population)

In [39]:
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=data["date"],
    y=data["case_cnt"],
    mode="lines",
    name="Actual",
    line=dict(width=2, color="#2a78d6"),
    hovertemplate="%{x|%b %d, %Y}<br>Cases: %{y:,}<extra></extra>",
))

fig.add_trace(go.Scatter(
    x=data["date"],
    y=cases_forecast,
    mode="lines",
    name="Forecast",
    line=dict(width=2, color="#e3a445", dash="dash"),
    hovertemplate="%{x|%b %d, %Y}<br>Cases Forecast: %{y:,}<extra></extra>",
))

fig.add_vline(
    x=train_test_split,
    line_width=2,
    line_dash="dot",
    line_color="#d62728",
    annotation_text="Train/Test cutoff",
    annotation_position="top left",
)

fig.update_layout(
    title="COVID-19 Cases Actual vs Forecast — US-NY",
    xaxis_title="Date",
    yaxis_title="Number of Cases",
    hovermode="x unified",
    template="plotly_white",
    plot_bgcolor="#fcfcfb",
    paper_bgcolor="#fcfcfb",
    xaxis=dict(showgrid=False, linecolor="#c3c2b7"),
    yaxis=dict(gridcolor="#e1e0d9", zeroline=False),
)

fig.show()

In [41]:
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=data["date"],
    y=data["death_cnt"],
    mode="lines",
    name="Actual",
    line=dict(width=2, color="#2a78d6"),
    hovertemplate="%{x|%b %d, %Y}<br>Deaths: %{y:,}<extra></extra>",
))

fig.add_trace(go.Scatter(
    x=data["date"],
    y=cases_forecast,
    mode="lines",
    name="Forecast",
    line=dict(width=2, color="#e3a445", dash="dash"),
    hovertemplate="%{x|%b %d, %Y}<br>Deaths Forecast: %{y:,}<extra></extra>",
))

fig.add_vline(
    x=train_test_split,
    line_width=2,
    line_dash="dot",
    line_color="#d62728",
    annotation_text="Train/Test cutoff",
    annotation_position="top left",
)

fig.update_layout(
    title="COVID-19 Deaths Actual vs Forecast — US-NY",
    xaxis_title="Date",
    yaxis_title="Number of Deaths",
    hovermode="x unified",
    template="plotly_white",
    plot_bgcolor="#fcfcfb",
    paper_bgcolor="#fcfcfb",
    xaxis=dict(showgrid=False, linecolor="#c3c2b7"),
    yaxis=dict(gridcolor="#e1e0d9", zeroline=False),
)

fig.show()